# Height-Weight Linear Regression

This notebook supports two explicitly separate modes:

- **original**: analyze the original CSV when it is supplied locally.
- **synthetic**: run an independently parameterized demonstration of the workflow.

Synthetic-mode results are illustrative only and are not a reproduction, verification, or confirmation of the historical metrics reported for the original 10,000-row dataset.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

## Choose the data source

The repository does not currently include the original CSV. The notebook therefore defaults to a synthetic demonstration.

To rerun the empirical analysis after obtaining the original source file, place it at `data/weight-height.csv` (or change `CSV_PATH`) and set `DATA_MODE = "original"`.

In [ ]:
DATA_MODE = "synthetic"  # choose "synthetic" or "original"
CSV_PATH = Path("data/weight-height.csv")

SYNTHETIC_SEED = 2026
SYNTHETIC_N = 1500

if DATA_MODE not in {"synthetic", "original"}:
    raise ValueError("DATA_MODE must be either 'synthetic' or 'original'.")

## Load original data or create an illustrative dataset

The synthetic data-generating process below is intentionally independent of the historical fitted intercept (-350.737192), slope (7.717288), and residual standard deviation (12.2198). Its only purpose is to provide plausible data for exercising the regression workflow.

In [ ]:
def _normalized_name(name):
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


def standardize_height_weight_columns(data):
    aliases = {
        "Height_Inches": {"height", "heightinches", "heightinch"},
        "Weight_Pounds": {"weight", "weightpounds", "weightpound", "weightlbs", "weightlb"},
    }

    normalized = {_normalized_name(column): column for column in data.columns}
    selected = {}

    for target, candidates in aliases.items():
        match = next((normalized[name] for name in candidates if name in normalized), None)
        if match is None:
            raise ValueError(
                f"Could not find a column for {target}. "
                f"Available columns: {list(data.columns)}"
            )
        selected[target] = match

    clean = data[[selected["Height_Inches"], selected["Weight_Pounds"]]].copy()
    clean.columns = ["Height_Inches", "Weight_Pounds"]

    clean["Height_Inches"] = pd.to_numeric(clean["Height_Inches"], errors="coerce")
    clean["Weight_Pounds"] = pd.to_numeric(clean["Weight_Pounds"], errors="coerce")
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < 3:
        raise ValueError("At least three valid height-weight observations are required.")

    return clean


def load_original_csv(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Original-data mode requested, but {path} was not found. "
            "Supply the source CSV or update CSV_PATH."
        )
    return standardize_height_weight_columns(pd.read_csv(path))


def make_synthetic_demo(n=SYNTHETIC_N, seed=SYNTHETIC_SEED):
    rng = np.random.default_rng(seed)

    # Independently chosen illustrative parameters.
    # These are not copied from or calibrated to the historical project fit.
    height = rng.normal(loc=67.0, scale=3.5, size=n)
    noise = rng.normal(loc=0.0, scale=20.0, size=n)
    weight = -185.0 + 5.3 * height + noise

    return pd.DataFrame(
        {
            "Height_Inches": height,
            "Weight_Pounds": weight,
        }
    )


if DATA_MODE == "original":
    df = load_original_csv(CSV_PATH)
    source_label = f"ORIGINAL CSV: {CSV_PATH}"
else:
    df = make_synthetic_demo()
    source_label = (
        "SYNTHETIC DEMONSTRATION: independently parameterized illustrative data; "
        "not a reproduction of the historical original-data results"
    )

print(source_label)
print(f"Observations used: {len(df):,}")
df.head(10)

## Descriptive statistics

Every result from this point forward applies only to the dataset identified by the printed source label above.

In [ ]:
df.describe()

## Scatter plot

The scatter plot visualizes the relationship in the currently loaded dataset.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["Height_Inches"], df["Weight_Pounds"], alpha=0.25)
plt.title(f"Height vs Weight — {DATA_MODE.title()} Mode")
plt.xlabel("Height (Inches)")
plt.ylabel("Weight (Pounds)")
plt.show()

## Manual regression calculations

The slope and intercept are computed directly from the currently loaded observations.

In [ ]:
x = df["Height_Inches"]
y = df["Weight_Pounds"]

x_mean = x.mean()
y_mean = y.mean()
slope_manual = ((x - x_mean) * (y - y_mean)).sum() / ((x - x_mean) ** 2).sum()
intercept_manual = y_mean - slope_manual * x_mean
pred_manual = intercept_manual + slope_manual * x
rss_manual = ((y - pred_manual) ** 2).sum()

print(source_label)
print(f"Mean Height: {x_mean:.4f}")
print(f"Mean Weight: {y_mean:.4f}")
print(f"Manual slope: {slope_manual:.6f}")
print(f"Manual intercept: {intercept_manual:.6f}")
print(f"Residual Sum of Squares: {rss_manual:.4f}")

## Regression using statsmodels

In [ ]:
X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
print(model.summary())

## Compact model results

These values are computed from the active dataset. In synthetic mode they describe only the illustrative data.

In [ ]:
slope_ci = model.conf_int().loc["Height_Inches"]
residual_standard_error = np.sqrt(model.ssr / model.df_resid)

results = pd.Series(
    {
        "data_mode": DATA_MODE,
        "observations": int(model.nobs),
        "intercept": model.params["const"],
        "slope": model.params["Height_Inches"],
        "r_squared": model.rsquared,
        "residual_sum_of_squares": model.ssr,
        "residual_standard_error": residual_standard_error,
        "slope_ci_95_low": slope_ci.iloc[0],
        "slope_ci_95_high": slope_ci.iloc[1],
        "slope_p_value": model.pvalues["Height_Inches"],
    }
)

print(source_label)
results

## Regression line

In [ ]:
order = np.argsort(x.to_numpy())

plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.2, label="Observed data")
plt.plot(
    x.to_numpy()[order],
    model.fittedvalues.to_numpy()[order],
    linewidth=2,
    label="Regression line",
)
plt.title(f"Height vs Weight with Regression Line — {DATA_MODE.title()} Mode")
plt.xlabel("Height (Inches)")
plt.ylabel("Weight (Pounds)")
plt.legend()
plt.show()

## Residual plot

Residuals should be examined for systematic patterns before relying on a simple linear model.

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

plt.figure(figsize=(8, 6))
plt.scatter(fitted, residuals, alpha=0.25)
plt.axhline(0, linestyle="--")
plt.title(f"Residuals vs Fitted Values — {DATA_MODE.title()} Mode")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.show()

## Historical original-data results

The original project recorded the following values for a 10,000-row dataset:

- intercept: **-350.737192**
- slope: **7.717288**
- R-squared: **0.8552**
- residual sum of squares: **1,492,934.8396**
- residual standard error: **12.2198**
- 95% slope confidence interval: **7.655028 to 7.779547**
- slope p-value: **< 0.001**

Because the source CSV is not currently in the repository, these values are historical reported results. They are **not** targets for synthetic mode and should only be called reproduced if `DATA_MODE = "original"` is run on the actual source data and independently yields them (within reasonable numerical tolerance).

## Interpretation

Interpret the fitted coefficients, model fit, confidence intervals, and diagnostics according to the active data source.

- In **original** mode, the output is an empirical rerun of the supplied CSV.
- In **synthetic** mode, the output demonstrates the mechanics of the analysis only.

Height alone should not be interpreted as a causal or complete explanation of weight; other biological, behavioral, measurement, and contextual factors can matter.